In [ ]:
import os
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from google import genai
import faiss

# ---------------------------------------
# GPU Detection (SentenceTransformers will use this!)
# ---------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using Device for Embeddings:", device)
if device == "cuda":
    print("GPU Model:", torch.cuda.get_device_name(0))

# ---------------------------------------
# Load Dataset
# ---------------------------------------
CSV_FILE = "medical_reports.csv"

if not os.path.exists(CSV_FILE):
    raise FileNotFoundError(f"Could not find the file at {CSV_FILE}. Please verify your file path.")

df = pd.read_csv(CSV_FILE)
print("Dataset loaded successfully. Shape:", df.shape)

# ---------------------------------------
# Convert CSV into Documents
# One document per patient
# ---------------------------------------
documents = []
for patient_id, group in df.groupby("Patient_ID"):
    report = f"""
Patient ID : {patient_id}
Name : {group.iloc[0]['Name']}
Age : {group.iloc[0]['Age']}
Gender : {group.iloc[0]['Gender']}

Medical Report
--------------
"""
    for _, row in group.iterrows():
        report += f"""
Test Name : {row['Test_Name']}
Result : {row['Result']} {row['Unit']}
Reference Range : {row['Reference_Range']}
Status : {row['Status']}
"""
    documents.append(report)

print("\nSample Document Processed:\n", documents[0])

# ---------------------------------------
# Load Embedding Model
# ---------------------------------------
print("\nLoading Embedding Model...")
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)

# ---------------------------------------
# Generate Embeddings
# ---------------------------------------
print("Generating Embeddings...")
embeddings = embedder.encode(
    documents,
    batch_size=16,
    convert_to_numpy=True,
    show_progress_bar=True
)
print("Embedding Shape:", embeddings.shape)

# ---------------------------------------
# Build FAISS Index (Using stable CPU tracking)
# ---------------------------------------
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print("Documents successfully indexed inside FAISS:", index.ntotal)

# ---------------------------------------
# Gemini API Initialization
# ---------------------------------------
# Best Practice: Use environment variables or enter it securely
API_KEY = os.getenv("GEMINI_API_KEY", "AIxxxxzaSyBxx5kpmmxxxxxxxxxxxxxxxxxxA6767676767")
client = genai.Client(api_key=API_KEY)

# ---------------------------------------
# Ask Gemini Function
# ---------------------------------------
def ask_gemini(question, top_k=2):
    q_embedding = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_embedding, top_k)

    # Reconstruct context safely
    context = "\n\n".join(documents[i] for i in indices[0] if i < len(documents))

    prompt = f"""
You are an experienced medical report assistant.
Below is the retrieved patient report.

{context}

Answer ONLY from the report.
If the information is unavailable, say:
"The requested information is not available in this report."

Never diagnose diseases.
Never prescribe medicine.
Explain medical terms in simple language.

Question:
{question}
"""
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
        return response.text
    except Exception as e:
        return f"An error occurred while generating content: {str(e)}"

# ---------------------------------------
# Chat Loop
# ---------------------------------------
print("\n" + "="*60)
print("Medical Report Assistant Ready")
print("="*60)

while True:
    question = input("\nAsk Question (type exit): ")
    if question.lower() == "exit":
        print("Exiting assistant loop.")
        break
        
    if not question.strip():
        continue

    answer = ask_gemini(question)
    print("\nGemini Answer:\n")
    print(answer)

Using Device for Embeddings: cuda
GPU Model: NVIDIA H200 MIG 1g.18gb
Dataset loaded successfully. Shape: (10, 10)

Sample Document Processed:
 
Patient ID : P001
Name : John Doe
Age : 45
Gender : Male

Medical Report
--------------

Test Name : Hemoglobin
Result : 10.5 g/dL
Reference Range : 13.5-17.5
Status : Low

Test Name : WBC
Result : 8200.0 cells/uL
Reference Range : 4000-11000
Status : Normal

Test Name : Platelets
Result : 180000.0 cells/uL
Reference Range : 150000-450000
Status : Normal

Test Name : Fasting Glucose
Result : 165.0 mg/dL
Reference Range : 70-99
Status : High


Loading Embedding Model...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 20423.97it/s]


Generating Embeddings...


Batches: 100%|████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.10it/s]


Embedding Shape: (3, 384)
Documents successfully indexed inside FAISS: 3

Medical Report Assistant Ready



Ask Question (type exit):  is he gonna die



Gemini Answer:

The requested information is not available in this report.
